# Otimização Inteligente do PNAE — Regras PNAE de Restrições para o Otimizador

**Aluna:** Daniela Barreto Araújo

**Orientador:** Renato Cividini Matthiesen

**Curso:** MBA em Data Science & Analytics para Operações

**Projeto:** Otimização inteligente do PNAE: integração de modelos preditivos e programação linear na alimentação escolar

Este notebook documenta a etapa de **ingestão, tratamento (ETL) e análise exploratória**
da base de preços dos alimentos da CONAB (Companhia Nacional de Abastecimento), servindo de
insumo para a etapa seguinte de modelagem por Programação Linear (Problema da Dieta).

---

## Objetivo do módulo
Traduzir as regras da **Resolução CD/FNDE nº 4, de 26 de fevereiro de 2026** (arts. 18 e 19) e a **exigência de 45 % de aquisição da Agricultura Familiar** (Lei nº 15.226/2025, que alterou a Lei nº 11.947/2009, art. 14, elevando o piso de 30 % para 45 %) em **objetos Python** estruturados que servirão como entrada direta para o modelo de Programação Linear (PL).

### Entradas esperadas
| Arquivo | Origem | Descrição |
|---|---|---|
| `pnae_alimentos_naoproibidos.csv` | `otimizacao_pnae_alimentos.ipynb` | Base nutricional com flag de restrição |
| `resumo_precos_referencia.csv` | `otimizacao_pnae_precos_conab.ipynb` | Preço de referência por alimento |
| `demanda_nutricional_escolas.csv` | `otimizacao_pnae_censo.ipynb` | Demanda nutricional por escola/grupo |

### Saídas geradas
| Arquivo | Uso no otimizador |
|---|---|
| `restricoes_pnae.json` | Parâmetros completos de restrição para o solver |
| `alimentos_elegíveis_otimizador.csv` | Dataset consolidado pronto para a PL |
| `relatorio_restricoes_pnae.txt` | Relatório legível para nutricionistas |


## Bloco 1 — Importações e configurações

In [2]:
import pandas as pd
import numpy as np
import json
import os
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

# ── Pastas padrão do projeto ───────────────────────────────────────────────
input_folder  = './data/input/'
output_folder = './data/output/'
os.makedirs(output_folder, exist_ok=True)

# ── Configurações de exibição ──────────────────────────────────────────────
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print(f"Módulo carregado em: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Pasta de saída: {os.path.abspath(output_folder)}")


Módulo carregado em: 2026-09-06 09:17:53
Pasta de saída: c:\Users\dba14\Documents\projeto\PNAE\data\output


## Bloco 2 — Carregamento das bases dos módulos anteriores

As três bases são geradas pelos notebooks anteriores do projeto.  
Este bloco faz a **validação de existência e integridade** antes de continuar.


In [3]:
# ── Caminhos dos arquivos ──────────────────────────────────────────────────
PATH_ALIMENTOS = output_folder + 'pnae_alimentos_naoproibidos.csv'
PATH_PRECOS    = output_folder + 'resumo_precos_referencia.csv'
PATH_DEMANDA   = output_folder + 'demanda_nutricional_escolas.csv'

def carregar_csv(path, sep=',', **kwargs):
    """Carrega CSV com validação de existência."""
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Arquivo não encontrado: {path}\n"
            f"Execute o notebook de origem antes de rodar este módulo."
        )
    df = pd.read_csv(path, sep=sep, low_memory=False, **kwargs)
    print(f"  ✓ {os.path.basename(path):50s} → {df.shape[0]:>6} linhas × {df.shape[1]:>3} colunas")
    return df

print("Carregando bases...")
df_alimentos = carregar_csv(PATH_ALIMENTOS)
df_precos    = carregar_csv(PATH_PRECOS)
df_demanda   = carregar_csv(PATH_DEMANDA, sep=';', decimal=',')

print("\nCarregamento concluído.")

# ═══════════════════════════════════════════════════════════════════════════
# CORREÇÃO (jul/2026): excluir da base do solver os alimentos marcados no
# Módulo 1 como NÃO precificáveis (df['pode_precificar'] == False).
# ─────────────────────────────────────────────────────────────────────────
# Causa-raiz: o Módulo 1 cria a coluna 'pode_precificar' (Bloco: "Filtro:
# Itens sem preço unitário CONAB possível") para sinalizar preparações
# compostas/receitas (ex.: "Abacaxi, banana e cenoura, suco natural (néctar),
# c/ açúcar refinado") que não têm preço de mercado/feira estabelecível. Essa
# coluna é usada corretamente pelo Módulo 3 (filtra 'df = df[df["pode_precificar"]
# == True]' antes do matching CONAB) — mas o Módulo 4 carregava
# 'pnae_alimentos_naoproibidos.csv' (saída COMPLETA e não filtrada do Módulo 1)
# sem aplicar o mesmo filtro. O item ficava sem preço real (NaN), e a imputação
# por mediana de categoria no solver.py acabava atribuindo um preço "artificial"
# a um alimento que nunca deveria ter entrado na base do otimizador.
# Correção: aplicar o mesmo filtro aqui, ANTES de qualquer outro processamento.
if 'pode_precificar' in df_alimentos.columns:
    n_antes = len(df_alimentos)
    excluidos_precificacao = df_alimentos.loc[
        df_alimentos['pode_precificar'] == False,
        [c for c in ['alimento', 'motivo_exclusao'] if c in df_alimentos.columns]
    ]
    df_alimentos = df_alimentos[df_alimentos['pode_precificar'] == True].copy()
    n_excluidos = n_antes - len(df_alimentos)
    print(f"✓ Filtro 'pode_precificar': {n_excluidos} alimento(s) removido(s) da base do "
          f"otimizador (preparações/receitas sem preço de mercado estabelecível).")
    if n_excluidos > 0:
        display(excluidos_precificacao.head(20))
else:
    print("⚠ Coluna 'pode_precificar' não encontrada em pnae_alimentos_naoproibidos.csv — "
          "nenhum item foi excluído por este critério. Verifique a versão do Módulo 1.")


Carregando bases...
  ✓ pnae_alimentos_naoproibidos.csv                    →    534 linhas ×  48 colunas
  ✓ resumo_precos_referencia.csv                       →    309 linhas ×  20 colunas
  ✓ demanda_nutricional_escolas.csv                    →   1021 linhas ×  17 colunas

Carregamento concluído.
✓ Filtro 'pode_precificar': 81 alimento(s) removido(s) da base do otimizador (preparações/receitas sem preço de mercado estabelecível).


,alimento,motivo_exclusao
1,"Abacaxi, banana e cenoura, suco natural (nécta...",preparacao_detectada:\bc\/\s
2,"Abacaxi, banana e cenoura, suco natural (nécta...",preparacao_detectada:\be\s+\w+
4,"Abacaxi, maracujá e caju, suco natural (néctar...",preparacao_detectada:\bc\/\s
5,"Abacaxi, melão e maracujá, suco natural (nécta...",preparacao_detectada:\bc\/\s
6,"Abacaxi, melão e maracujá, suco natural (nécta...",preparacao_detectada:\be\s+\w+
9,"Abacaxi, suco natural (néctar), c/ açúcar refi...",preparacao_detectada:\bc\/\s
10,"Abacaxi, suco natural (néctar), s/ açúcar",preparacao_detectada:\b(suco|néctar|vitamina|s...
24,"Acerola, suco natural (néctar), c/ açúcar refi...",preparacao_detectada:\bc\/\s
25,"Acerola, suco natural (néctar), s/ açúcar",preparacao_detectada:\b(suco|néctar|vitamina|s...
26,Achocolatado em pó diet,preparacao_detectada:\b(suco|néctar|vitamina|s...


## Bloco 3 — Definição das Regras PNAE (Resolução CD/FNDE nº 4/2026)

Este bloco é o **núcleo do módulo**: define todas as constantes e estruturas de restrição
em um único dicionário Python (`REGRAS_PNAE`) que será serializado como JSON para o otimizador.

> **Fonte legal:** Resolução CD/FNDE nº 4, de 26 de fevereiro de 2026, Arts. 18 e 19.  
> **Agricultura Familiar:** Lei nº 11.947/2009, Art. 14 + Resolução CD/FNDE nº 4, de 26 de fevereiro de 2026: **45 %**.


In [4]:
# ═══════════════════════════════════════════════════════════════════════════
# CONSTANTES DE REGRAS — Resolução CD/FNDE nº 4/2026
# Altere SOMENTE aqui caso a resolução seja atualizada.
# ═══════════════════════════════════════════════════════════════════════════

REGRAS_PNAE = {

    # ── METADADOS ────────────────────────────────────────────────────────────
    "_metadata": {
        "fonte_legal"  : "Resolução CD/FNDE nº 4, de 26 de fevereiro de 2026",
        "arts_referencia": ["Art. 18", "Art. 19"],
        "af_fonte"     : "Lei nº 15.226/2025, que altera a Lei nº 11.947/2009, Art. 14, elevando o piso de 30 % para 45 %, em vigor desde 1º/jan/2026",
        "gerado_em"    : datetime.now().isoformat(),
        "versao"       : "1.0.0",
    },

    # ── 1. PERCENTUAL NUTRICIONAL MÍNIMO (% das necessidades diárias) ────────
    # Chave: identificador do grupo (usado em demanda_nutricional_escolas.csv)
    # Valor: percentual mínimo de cobertura das necessidades nutricionais
    "percentual_nutricional_minimo": {
        # Creches
        "creche_parcial"         : 0.30,   # ≥ 30 % em ≥ 2 refeições
        "creche_integral"        : 0.70,   # ≥ 70 % em ≥ 3 refeições (incl. indígenas/quilombolas)
        # Escolas indígenas, quilombolas e outros PCT (exceto creches)
        "indigena_quilombola_pct": 0.30,   # 30 % por refeição ofertada
        # Educação básica período parcial
        "parcial_1_refeicao"     : 0.20,   # ≥ 20 % com 1 refeição
        "parcial_2_refeicoes"    : 0.30,   # ≥ 30 % com 2+ refeições
        # Educação integral
        "integral"               : 0.70,   # ≥ 70 % em ≥ 3 refeições
    },

    # ── 2. FRUTAS, LEGUMES E VERDURAS (FLV) — quantidade semanal (g/aluno) ──
    "flv_semana_gramas": {
        "parcial" : 280,   # mínimo 280 g/aluno/semana
        "integral": 520,   # mínimo 520 g/aluno/semana
    },

    # Frequência mínima (dias por semana de 5 dias letivos)
    "flv_frequencia_minima_dias": {
        "frutas_in_natura_parcial" : 2,   # ≥ 2 dias/semana
        "frutas_in_natura_integral": 4,   # ≥ 4 dias/semana
        "legumes_verduras_parcial" : 3,   # ≥ 3 dias/semana
        "legumes_verduras_integral": 5,   # ≥ 5 dias/semana
    },

    # Observação: bebidas à base de frutas NÃO substituem fruta in natura
    "bebida_fruta_substitui_fruta_in_natura": False,

    # ── 3. MICRONUTRIENTES — frequência mínima semanal (dias) ────────────────
    "micronutrientes_frequencia_minima_dias": {
        # Ferro heme (carne, vísceras, peixe etc.)
        "ferro_heme"    : 4,   # ≥ 4 dias/semana
        # Se substituir por ferro não-heme, deve haver vitamina C simultânea
        "ferro_nao_heme_requer_vitamina_c": True,
        # Vitamina A (fígado, abóbora, cenoura, folhas verde-escuras etc.)
        "vitamina_a"    : 3,   # ≥ 3 dias/semana
    },

    # ── 4. LIMITES MÁXIMOS DE FREQUÊNCIA (vezes/mês ou vezes/semana) ─────────
    "limites_maximos": {

        # Produtos cárneos (salsicha, linguiça, mortadela etc.)
        "produtos_carneos_vezes_mes"          : 2,

        # Alimentos em conserva
        "conservas_vezes_mes"                 : 1,

        # Líquidos lácteos com aditivos ou adoçados
        "lacteos_aditivados_parcial_vezes_mes": 1,
        "lacteos_aditivados_integral_vezes_mes": 2,

        # Biscoito / bolacha / pão / bolo
        "biscoito_pao_bolo_parcial_1ref_vezes_semana"   : 2,
        "biscoito_pao_bolo_parcial_2ref_vezes_semana"   : 3,
        "biscoito_pao_bolo_integral_3ref_vezes_semana"  : 7,

        # Doces
        "doces_vezes_mes"                     : 1,

        # Preparações doces (sobremesa, mingau adoçado etc.)
        "preparacoes_doces_parcial_vezes_mes" : 2,
        "preparacoes_doces_integral_vezes_semana": 1,

        # Margarina / creme vegetal
        "margarina_parcial_vezes_mes"         : 2,
        "margarina_integral_vezes_semana"     : 1,
    },

    # ── 5. PROIBIÇÕES ABSOLUTAS ───────────────────────────────────────────────
    "proibicoes": {
        # Gordura trans industrializada — proibida em TODOS os cardápios
        "gordura_trans_industrializada": True,

        # Restrições para crianças até 3 anos (creche parcial/integral)
        "menores_3_anos": {
            "ultraprocessados"          : True,   # proibido
            "adicao_acucar"             : True,   # proibido
            "adicao_mel"                : True,   # proibido
            "adicao_adocantes"          : True,   # proibido
        },
    },

    # ── 6. LIMITES DE MACRONUTRIENTES (% do VET) — para > 3 anos ────────────
    # AJUSTE (set/2026): Art. 19, caput, da Resolução CD/FNDE nº 4/2026 usa o
    # verbo "recomenda-se" — os três itens abaixo são RECOMENDAÇÕES, não
    # proibições legais absolutas (diferente da seção "proibicoes" acima, que
    # trata de vedações efetivas, ex.: gordura trans industrializada).
    #
    # LIMITAÇÃO METODOLÓGICA DOCUMENTADA (set/2026): os itens II e III do
    # Art. 19 (gorduras totais e gordura saturada, em % VET) NÃO são
    # implementados como restrição ativa em nenhum bloco deste notebook nem
    # no guia de integração do Bloco 11. A base de composição nutricional
    # (TACO/TBCA, tratada no Módulo 1) traz apenas "Lipídios" agregados, sem
    # discriminar o tipo de gordura (saturada, insaturada, trans não
    # industrializada) — sem esse detalhamento não é possível calcular o
    # %VET de gordura saturada com confiabilidade. Os valores ficam
    # registrados aqui como referência normativa e para trabalho futuro
    # (ex.: enriquecer a base com uma fonte de composição de ácidos graxos).
    #
    # Item I (açúcar simples adicionado) também não é calculado como %VET
    # real (a base não discrimina açúcares por tipo dentro de cada
    # alimento/preparação). Como proxy parcial, o grupo operacional
    # "acucar_simples_adicionado" (seção 10 abaixo) identifica os
    # INGREDIENTES que são açúcar adicionado puro (açúcar cristal, açúcar
    # refinado, açúcar mascavo, mel de abelha, melado, rapadura) — não
    # açúcares naturalmente presentes em frutas e outros alimentos, conforme
    # a distinção do Art. 19, I, que fala em açúcar "adicionado".
    "limites_macronutrientes_pct_vet": {
        "acucar_simples_adicionado_max": 0.07,    # Art. 19, I — recomendação; ≤7% VET;
                                                     # ver nota acima (proxy parcial, não %VET real)
        "gorduras_totais_min"          : 0.15,    # Art. 19, II — recomendação; NÃO implementado (ver nota acima)
        "gorduras_totais_max"          : 0.30,    # Art. 19, II — recomendação; NÃO implementado (ver nota acima)
        "gordura_saturada_max"         : 0.07,    # Art. 19, III — recomendação; NÃO implementado (ver nota acima)
    },

    # ── 7. LIMITES DE SÓDIO (mg) e SAL (g) per capita ────────────────────────
    "limites_sodio": {
        "parcial_1_refeicao" : {"sodio_mg": 600,  "sal_g": 1.5},
        "parcial_2_refeicoes": {"sodio_mg": 800,  "sal_g": 2.0},
        "integral_3_refeicoes": {"sodio_mg": 1400, "sal_g": 3.5},
    },

    # ── 8. DIVERSIDADE DE ALIMENTOS IN NATURA / MINIMAMENTE PROCESSADOS ──────
    # Quantidade mínima de alimentos DIFERENTES por semana
    "diversidade_minima_alimentos": {
        "cardapio_1_refeicao_ou_20pct" : 10,
        "cardapio_2_refeicoes_ou_30pct": 14,
        "cardapio_3_refeicoes_ou_70pct": 23,
    },

    # ── 9. AGRICULTURA FAMILIAR ───────────────────────────────────────────────
    "agricultura_familiar": {
        "percentual_minimo_legal"   : 0.45,   # CORREÇÃO (set/2026): citação legal ajustada.
        # O piso de 45 % vem da Lei nº 15.226/2025, que ALTEROU a Lei nº 11.947/2009
        # (Art. 14) e elevou o mínimo de aquisição da Agricultura Familiar de 30 % para
        # 45 %, em vigor desde 1º/jan/2026. A Lei nº 11.947/2009 sozinha, na redação
        # original, previa apenas 30 %.
        "percentual_meta_fnde_2026" : 0.45,   # Meta FNDE 2026 (usar como default)
        "base_calculo"              : "recursos_repassados_fnde",
        "prioridade_compra"         : [
            "associacoes_cooperativas_locais",
            "agricultores_individuais_municipio",
            "agricultores_individuais_estado",
            "agricultores_individuais_territorio",
        ],
        "documentos_habilitacao"    : ["DAP", "CAF"],  # Declaração de Aptidão ao PRONAF / Cadastro AF
        "observacao"                : (
            "O percentual de 45 % é adotado como meta padrão neste projeto. "
            "O solver deve garantir que pelo menos 45 % do custo total de "
            "aquisição seja de fornecedores com flag 'agricultura_familiar=True'."
        ),
    },

    # ── 10. GRUPOS DE ALIMENTOS — categorias operacionais ─────────────────────
    # Usadas para mapear restrições de frequência às colunas do dataset
    "grupos_operacionais": {
        "frutas_in_natura"       : ["fruta", "abacaxi", "banana", "laranja", "mamão", "melão",
                                    "manga", "maçã", "pera", "uva", "goiaba", "acerola",
                                    "caju", "caqui", "carambola", "figo", "jaca", "melancia",
                                    "morango", "nectarina", "pêssego", "tangerina"],
        "legumes_verduras"       : ["legume", "verdura", "alface", "couve", "espinafre",
                                    "brócolis", "cenoura", "beterraba", "abobrinha", "berinjela",
                                    "chuchu", "jiló", "maxixe", "pepino", "pimentão", "quiabo",
                                    "tomate", "vagem", "mandioquinha", "batata-doce"],
        "ferro_heme"             : ["carne", "frango", "peixe", "atum", "sardinha", "fígado",
                                    "víscera", "miúdo"],
        "ferro_nao_heme"         : ["feijão", "lentilha", "grão-de-bico", "ervilha", "soja",
                                    "espinafre", "couve", "brócolis", "tofu"],
        "vitamina_a"             : ["fígado", "cenoura", "abóbora", "batata-doce", "mamão",
                                    "manga", "espinafre", "couve", "brócolis", "ovo"],
        "produtos_carneos"       : ["salsicha", "linguiça", "mortadela", "presunto", "apresuntado",
                                    "carne processada", "embutido"],
        "conservas"              : ["conserva", "enlatado", "milho enlatado", "ervilha enlatada",
                                    "atum em lata", "sardinha em lata"],
        "lacteos_aditivados"     : ["achocolatado", "iogurte adoçado", "bebida láctea", "leite com"],
        "biscoito_pao_bolo"      : ["biscoito", "bolacha", "pão", "bolo", "torrada", "wafer"],
        "doces"                  : ["doce", "marmelada", "geleia", "chocolate", "bombom", "caramelo"],
        "preparacoes_doces"      : ["sobremesa", "pudim", "moussse", "compota", "mingau adoçado",
                                    "canjica"],
        "margarina_creme_vegetal": ["margarina", "creme vegetal"],
        "ultraprocessados"       : ["ultraprocessado", "aquisição proibida"],
        # AJUSTE (set/2026): grupo criado para dar suporte ao Art. 19, I
        # (açúcar simples adicionado, recomendação de ≤7% VET — ver seção 6
        # acima). Lista fechada por INGREDIENTE (não captura preparações que
        # contenham açúcar como componente, apenas os itens de açúcar/mel/
        # melado/rapadura em si), a pedido da autora. NÃO inclui frutas nem
        # qualquer alimento com açúcar naturalmente presente — apenas os
        # itens que SÃO açúcar/adoçante adicionado puro.
        "acucar_simples_adicionado": ["açúcar cristal", "açúcar refinado", "açúcar mascavo",
                                    "mel de abelha", "melado", "rapadura"],
    },
}

print("✓ Dicionário REGRAS_PNAE definido com sucesso.")
print(f"  Seções: {[k for k in REGRAS_PNAE.keys() if not k.startswith('_')]}")


✓ Dicionário REGRAS_PNAE definido com sucesso.
  Seções: ['percentual_nutricional_minimo', 'flv_semana_gramas', 'flv_frequencia_minima_dias', 'bebida_fruta_substitui_fruta_in_natura', 'micronutrientes_frequencia_minima_dias', 'limites_maximos', 'proibicoes', 'limites_macronutrientes_pct_vet', 'limites_sodio', 'diversidade_minima_alimentos', 'agricultura_familiar', 'grupos_operacionais']


## Bloco 4 — Enriquecimento do dataset de alimentos com flags de restrição

Este bloco mapeia cada alimento para os grupos operacionais definidos no Bloco 3 e adiciona
colunas booleanas que o otimizador usará como coeficientes nas restrições.


In [5]:
# ── Normalização auxiliar ─────────────────────────────────────────────────
import unicodedata, re

def normalizar(texto):
    """Remove acentos e converte para minúsculas para matching robusto.

    CORREÇÃO (set/2026): colapsa espaços múltiplos em um único espaço.
    Causa-raiz do bug: pontuação (ex.: vírgula em "Açúcar, cristal") é
    substituída por espaço, mas espaços consecutivos não eram colapsados —
    o resultado 'acucar  cristal' (2 espaços) nunca continha a chave já
    normalizada 'acucar cristal' (1 espaço) como substring. Qualquer
    palavra-chave de 2+ palavras falhava silenciosamente contra qualquer
    alimento cujo nome tivesse vírgula entre os mesmos termos (padrão comum
    no dataset: "Alimento, qualificador, preparo"). Confirmado com dados
    reais (set/2026): resolve a identificação de 'açúcar cristal/mascavo/
    refinado' e 'mel de abelha' (grupo 'acucar_simples_adicionado') e faz
    'Ervilha, enlatada, drenada' ser corretamente capturada por 'conservas'
    — hoje escapava do limite de frequência de conservas por este motivo.
    """
    if not isinstance(texto, str):
        return ""
    texto = unicodedata.normalize('NFD', texto)
    texto = ''.join(c for c in texto if unicodedata.category(c) != 'Mn')
    texto = re.sub(r'[^a-z0-9 ]', ' ', texto.lower())
    texto = re.sub(r'\s+', ' ', texto)   # CORREÇÃO (set/2026): colapsa espaços múltiplos
    return texto.strip()

# ── Coluna base de busca (para os flags de grupo operacional) ────────────
col_alimento = 'alimento'  # ajuste se o nome da coluna diferir no seu dataset

df_alimentos['_nome_normalizado'] = df_alimentos[col_alimento].apply(normalizar)

# ── Função: verificar se alimento pertence a grupo ────────────────────────
# CORREÇÃO (set/2026): PALAVRAS_MATCH_EXATO — palavras-chave curtas/genéricas
# que, por substring puro, capturam falsos positivos dentro de OUTRAS
# palavras. Achado concreto (auditoria set/2026, Bloco 4B): a palavra-chave
# 'ovo' (grupo vitamina_a) casava dentro de 'ovos' em "Macarrão, trigo, cru,
# com ovos" — um prato de massa sem relação nutricional real com a
# recomendação de vitamina A do Art. 19. Para as palavras listadas abaixo,
# a correspondência passa a exigir fronteira de palavra (regex \b). As
# demais palavras-chave (maioria) continuam usando substring, como antes —
# essa é uma correção pontual, não uma mudança geral da estratégia de
# matching (que exigiria reauditoria completa dos 14 grupos).
PALAVRAS_MATCH_EXATO = {'ovo'}

def mapear_grupo(nome_norm, palavras_chave):
    """Retorna True se alguma palavra-chave aparecer no nome normalizado.
    Por padrão, correspondência por substring (comportamento original).
    Para palavras em PALAVRAS_MATCH_EXATO, exige fronteira de palavra
    (\b) para evitar falsos positivos — ver CORREÇÃO (set/2026) acima."""
    for p in palavras_chave:
        p_norm = normalizar(p)
        if p_norm in PALAVRAS_MATCH_EXATO:
            if re.search(rf'\b{re.escape(p_norm)}\b', nome_norm):
                return True
        elif p_norm in nome_norm:
            return True
    return False

# ── Criar flags booleanas por grupo operacional ───────────────────────────
grupos = REGRAS_PNAE["grupos_operacionais"]

for grupo, palavras in grupos.items():
    df_alimentos[f'flag_{grupo}'] = df_alimentos['_nome_normalizado'].apply(
        lambda nome: mapear_grupo(nome, palavras)
    )

# ═══════════════════════════════════════════════════════════════════════════
# ═══════════════════════════════════════════════════════════════════════════
# AJUSTE (set/2026): identificação dos alimentos capturados pelo novo grupo
# 'acucar_simples_adicionado' (Art. 19, I — recomendação de ≤7% VET; ver
# REGRAS_PNAE["limites_macronutrientes_pct_vet"], seção 6). Exibido aqui
# para VALIDAÇÃO MANUAL antes de qualquer uso desse flag em restrição do
# solver — a lista de palavras-chave é curta e específica (açúcar cristal,
# açúcar refinado, açúcar mascavo, mel de abelha, melado, rapadura), mas
# como o matching ainda é por substring (mesma técnica dos demais grupos,
# ver achado de auditoria pendente sobre colisões de substring), os itens
# abaixo devem ser conferidos manualmente contra o dataset real.
# ═══════════════════════════════════════════════════════════════════════════
if 'flag_acucar_simples_adicionado' in df_alimentos.columns:
    n_acucar_add = df_alimentos['flag_acucar_simples_adicionado'].sum()
    print(f"\n── Identificação: alimentos = açúcar simples adicionado (Art. 19, I) ──────")
    print(f"  {n_acucar_add} item(ns) identificado(s). Revisar a lista abaixo manualmente:")
    if n_acucar_add > 0:
        display(
            df_alimentos.loc[
                df_alimentos['flag_acucar_simples_adicionado'],
                [c for c in [col_alimento, 'categoria'] if c in df_alimentos.columns]
            ].drop_duplicates()
        )

# CORREÇÃO (jul/2026): suco e polpa de frutas NÃO contam como fruta in natura
# ─────────────────────────────────────────────────────────────────────────
# Causa-raiz: 'flag_frutas_in_natura' é construída por matching de palavra-chave
# (ex.: 'abacaxi', 'manga') sobre o nome do alimento. Isso capturava incorretamente
# produtos como "Abacaxi, polpa, congelada" ou "Acerola, suco concentrado", pois o
# nome da fruta aparece no texto mesmo quando o produto é suco/polpa/néctar — que
# NÃO satisfazem a exigência de frutas in natura (Art. 19; ver também
# REGRAS_PNAE["bebida_fruta_substitui_fruta_in_natura"] = False, já documentado
# mas até então não aplicado como filtro efetivo).
# Correção: qualquer produto cuja categoria (Módulo 1) seja 'sucos_bebidas' OU cujo
# nome normalizado contenha um termo de suco/polpa/néctar/bebida é excluído do flag
# 'flag_frutas_in_natura', independentemente de conter o nome de uma fruta.
# ═══════════════════════════════════════════════════════════════════════════

TERMOS_BEBIDA_FRUTA = ['suco', 'polpa', 'nectar', 'refresco', 'bebida', 'refrigerante']

df_alimentos['_e_bebida_fruta'] = df_alimentos['_nome_normalizado'].apply(
    lambda nome: any(termo in nome for termo in TERMOS_BEBIDA_FRUTA)
)

if 'categoria' in df_alimentos.columns:
    df_alimentos['_e_bebida_fruta'] = (
        df_alimentos['_e_bebida_fruta'] | (df_alimentos['categoria'] == 'sucos_bebidas')
    )

n_excluidos_fruta = (df_alimentos['flag_frutas_in_natura'] & df_alimentos['_e_bebida_fruta']).sum()
df_alimentos['flag_frutas_in_natura'] = (
    df_alimentos['flag_frutas_in_natura'] & ~df_alimentos['_e_bebida_fruta']
)

print(f"✓ Correção suco/polpa aplicada: {n_excluidos_fruta} produto(s) removido(s) "
      f"de 'flag_frutas_in_natura' (eram suco/polpa/néctar classificados como fruta).")
if n_excluidos_fruta > 0:
    display(
        df_alimentos.loc[df_alimentos['_e_bebida_fruta'], [col_alimento, 'categoria']]
        .drop_duplicates().head(20)
    )

# ═══════════════════════════════════════════════════════════════════════════
# CORREÇÃO (jul/2026): join com o Módulo 3 (preços/AF)
# ─────────────────────────────────────────────────────────────────────────
# Causa-raiz do bug anterior (preço e AF sempre zerados):
#   1. O Módulo 3 exporta a chave 'produto_busca' e a flag 'is_af' — NUNCA
#      uma coluna chamada 'agricultura_familiar'. O código antigo procurava
#      por esse nome exato em df_precos e nunca o encontrava.
#   2. 'produto_busca' (Módulo 3) é derivado de 'produto_base' (Módulo 1),
#      mas 'produto_base' não estava na lista de chaves de join candidatas
#      — o fallback caía em 'alimento' (texto longo, ex.: "Abacaxi cru"),
#      que não corresponde ao texto normalizado do Módulo 3.
# Correção: normalizamos 'produto_base' (lado alimentos) e 'produto_busca'
# (lado preços) com a MESMA função normalizar() e fazemos o join por essa
# chave comum, trazendo preço, AF e metadados de confiança em uma única
# operação — usada tanto para a flag de AF quanto para o preço (Bloco 7/8).
# ═══════════════════════════════════════════════════════════════════════════

col_produto_alimentos = next(
    (c for c in ['produto_base', 'produto_busca'] if c in df_alimentos.columns),
    None
)

if col_produto_alimentos is None:
    raise KeyError(
        "Nenhuma coluna de produto-base ('produto_base'/'produto_busca') "
        "encontrada em pnae_alimentos_naoproibidos.csv. Verifique se o "
        "Módulo 1 (otimizacao_pnae_alimentos) foi executado com a versão "
        "que gera a coluna 'produto_base' (Bloco 8.2)."
    )

if 'produto_busca' not in df_precos.columns:
    raise KeyError(
        "Coluna 'produto_busca' não encontrada em resumo_precos_referencia.csv. "
        "Verifique a versão do Módulo 3 (otimizacao_pnae_precos_conab)."
    )

df_alimentos['_chave_join'] = df_alimentos[col_produto_alimentos].apply(normalizar)
df_precos['_chave_join']    = df_precos['produto_busca'].apply(normalizar)

# ═══════════════════════════════════════════════════════════════════════════
# CORREÇÃO (jul/2026): preservar 'produto_busca' explicitamente no dataset
# ─────────────────────────────────────────────────────────────────────────
# Causa-raiz de um bug correlato: o Bloco 8 (Dataset consolidado) tenta manter a
# coluna 'produto_busca' em 'cols_controle', mas ela NUNCA existia em df_alimentos
# como coluna nomeada (apenas '_chave_join', de uso interno) — então era sempre
# descartada silenciosamente do CSV exportado (alimentos_elegiveis_otimizador.csv).
# Isso quebra (a) o join defensivo de preço/AF feito no solver.py, e (b) o ajuste
# de sazonalidade mensal (aplicar_sazonalidade), que depende de 'produto_busca'
# para casar com sazonalidade_por_produto.csv — SEM esta coluna, a sazonalidade
# nunca era aplicada (fator sempre 1.0), mesmo quando os dados existiam.
# Correção: expor 'produto_busca' = mesma chave usada no join (produto_base já
# normalizado da mesma forma que o Módulo 3), preservando rastreabilidade.
# ═══════════════════════════════════════════════════════════════════════════
df_alimentos['produto_busca'] = df_alimentos[col_produto_alimentos].astype(str).str.strip()

cols_precos_join = ['_chave_join', 'preco_kg_referencia', 'is_af', 'confianca_af']
cols_precos_join = [c for c in cols_precos_join if c in df_precos.columns or c == '_chave_join']
df_precos_join = df_precos[cols_precos_join].drop_duplicates(subset='_chave_join')

n_antes_cols = df_alimentos.shape[1]
df_alimentos = df_alimentos.merge(df_precos_join, on='_chave_join', how='left')

# ── Flag booleana de Agricultura Familiar ─────────────────────────────────
# is_af pode vir como True / False / 'incerto' (ver Módulo 3, Bloco 8).
# Só tratamos como AF quando confirmado (True); 'incerto' e ausência viram False,
# mas 'confianca_af' fica disponível no dataset para auditoria/sensibilidade.
if 'is_af' in df_alimentos.columns:
    df_alimentos['agricultura_familiar'] = df_alimentos['is_af'].apply(
        lambda v: bool(v) if isinstance(v, (bool, int)) or str(v).lower() == 'true' else False
    )
else:
    df_alimentos['agricultura_familiar'] = False

# ── Diagnóstico do join ────────────────────────────────────────────────────
n_total       = len(df_alimentos)
n_com_preco   = df_alimentos['preco_kg_referencia'].notna().sum() if 'preco_kg_referencia' in df_alimentos.columns else 0
n_af          = df_alimentos['agricultura_familiar'].sum()

print(f"── Join Módulo 1 → Módulo 3 (preço + AF) ──────────────────────────────")
print(f"  Chave de join utilizada       : '{col_produto_alimentos}' × 'produto_busca' (normalizadas)")
print(f"  Alimentos com preço CONAB     : {n_com_preco:>4} / {n_total} ({n_com_preco/n_total*100:.1f} %)")
print(f"  Alimentos classificados AF    : {n_af:>4} / {n_total} ({n_af/n_total*100:.1f} %)")
if n_com_preco == 0:
    print("  ⚠ ALERTA: nenhum alimento recebeu preço. Verifique se")
    print("    resumo_precos_referencia.csv foi gerado pela versão atual do Módulo 3")
    print("    e se 'produto_base'/'produto_busca' realmente descrevem os mesmos itens.")

# ── Estatísticas de cobertura das flags de grupo operacional ─────────────
print("\n── Cobertura dos grupos operacionais ──────────────────────────────────")
for grupo in grupos.keys():
    col = f'flag_{grupo}'
    n = df_alimentos[col].sum()
    pct = n / len(df_alimentos) * 100
    print(f"  {grupo:35s}: {n:>4} alimentos ({pct:.1f} %)")

print(f"  {'agricultura_familiar':35s}: {n_af:>4} alimentos ({n_af/n_total*100:.1f} %)")

# ═══════════════════════════════════════════════════════════════════════════
# AJUSTE (jul/2026) — impacto da mudança no Módulo 3 (Bloco 12C / produtos_aptos_solver.csv)
# ─────────────────────────────────────────────────────────────────────────
# O Módulo 3 passou a excluir explicitamente, de resumo_precos_referencia.csv,
# os alimentos que não têm NENHUM preço de referência (nem CONAB, nem
# planilha manual). Antes desta correção, este notebook mantinha esses itens
# no dataset (join à esquerda → preco_kg_referencia == NaN) e deixava a
# decisão de tratá-los para o Módulo 5, que os IMPUTAVA com a mediana de
# preço da categoria (otimizacao_pnae_solver.py, preparar_dataset_solver) —
# ou seja, um alimento sem preço real acabava entrando no solver com um
# custo artificial, como se fosse um candidato válido.
#
# Correção: alinhado ao mesmo princípio já aplicado a proibidos e a
# is_ingrediente, um alimento sem preço de referência é retirado da base do
# otimizador AQUI, na origem, em vez de seguir adiante para ser imputado.
# `produtos_sem_preco.csv` (Módulo 3) continua sendo o registro oficial e
# auditável desses itens — nenhuma lista nova é criada neste notebook.
n_antes_preco = len(df_alimentos)
alimentos_sem_preco = df_alimentos.loc[
    df_alimentos['preco_kg_referencia'].isna(),
    [c for c in [col_alimento, 'produto_busca'] if c in df_alimentos.columns]
]
df_alimentos = df_alimentos[df_alimentos['preco_kg_referencia'].notna()].copy()
n_removidos_sem_preco = n_antes_preco - len(df_alimentos)

print(f"\n✓ Filtro 'preco_kg_referencia': {n_removidos_sem_preco} alimento(s) removido(s) da base "
      f"do otimizador (sem preço em nenhuma fonte CONAB nem na planilha manual do Módulo 3).")
if n_removidos_sem_preco > 0:
    display(alimentos_sem_preco.head(20))



── Identificação: alimentos = açúcar simples adicionado (Art. 19, I) ──────
  6 item(ns) identificado(s). Revisar a lista abaixo manualmente:


,alimento,categoria
27,"Açúcar, cristal",doces_acucares
28,"Açúcar, mascavo",doces_acucares
29,"Açúcar, refinado",doces_acucares
356,"Mel, de abelha",doces_acucares
357,Melado,doces_acucares
477,Rapadura,doces_acucares


✓ Correção suco/polpa aplicada: 8 produto(s) removido(s) de 'flag_frutas_in_natura' (eram suco/polpa/néctar classificados como fruta).


,alimento,categoria
7,"Abacaxi, polpa, congelada",sucos_bebidas
8,"Abacaxi, polpa, congelada",sucos_bebidas
20,"Açaí, polpa, congelada",sucos_bebidas
23,"Acerola, polpa, congelada",sucos_bebidas
32,Água de coco,sucos_bebidas
96,Café,sucos_bebidas
97,Café solúvel capuccino,sucos_bebidas
99,"Cajá, polpa, congelada",sucos_bebidas
100,"Cajá, polpa, congelada",sucos_bebidas
103,"Caju, polpa, congelada",sucos_bebidas


── Join Módulo 1 → Módulo 3 (preço + AF) ──────────────────────────────
  Chave de join utilizada       : 'produto_base' × 'produto_busca' (normalizadas)
  Alimentos com preço CONAB     :  317 / 453 (70.0 %)
  Alimentos classificados AF    :   84 / 453 (18.5 %)

── Cobertura dos grupos operacionais ──────────────────────────────────
  frutas_in_natura                   :   58 alimentos (12.8 %)
  legumes_verduras                   :   28 alimentos (6.2 %)
  ferro_heme                         :   61 alimentos (13.5 %)
  ferro_nao_heme                     :   26 alimentos (5.7 %)
  vitamina_a                         :   27 alimentos (6.0 %)
  produtos_carneos                   :   11 alimentos (2.4 %)
  conservas                          :   12 alimentos (2.6 %)
  lacteos_aditivados                 :    1 alimentos (0.2 %)
  biscoito_pao_bolo                  :   17 alimentos (3.8 %)
  doces                              :   21 alimentos (4.6 %)
  preparacoes_doces                  :    1

,alimento,produto_busca
4,"Abadejo, filé, congelado, cru",abadejo file
5,"Abiu, cru",abiu
23,"Aipo, cru",aipo
28,"Alfavaca, crua",alfavaca
33,"Almôndega, frango, crua",almondega frango
35,"Amendoim, grão, cru",amendoim grao
38,"Arroz, farelo",arroz farelo
45,"Atum, fresco, cru",atum fresco
52,Bacuri,bacuri
54,"Banana, doce em barra",banana doce em barra


## Bloco 4B — Auditoria: mapeamento de colisões de substring entre grupos operacionais (item 1.3)

**Diagnóstico apenas — nenhuma alteração é aplicada aqui.** O matching de `mapear_grupo()` (Bloco 4) é por substring simples, sem proteção de borda de palavra. Já identificamos manualmente um caso (`"doce"` capturando `"Batata, doce, crua"` dentro do grupo `doces`). Este bloco varre **todos** os pares de `flag_*` e lista, para cada par com sobreposição, os alimentos concretos capturados por ambos — para decidirmos, com o dado real, quais pares justificam correção (como a já feita em `flag_frutas_in_natura`) e quais são sobreposições legítimas (ex.: um alimento pode ser corretamente `ferro_heme` E `vitamina_a` ao mesmo tempo).

In [6]:
# ── Mapeamento de colisões de substring entre grupos operacionais (diagnóstico) ──
# Não altera 'grupos_operacionais' nem nenhum flag existente. Apenas relata.

flag_cols = [c for c in df_alimentos.columns if c.startswith('flag_')]

overlaps = []
for i in range(len(flag_cols)):
    for j in range(i + 1, len(flag_cols)):
        c1, c2 = flag_cols[i], flag_cols[j]
        n = (df_alimentos[c1] & df_alimentos[c2]).sum()
        if n > 0:
            overlaps.append({
                'grupo_1': c1.replace('flag_', ''),
                'grupo_2': c2.replace('flag_', ''),
                'n_itens': n,
            })

df_overlaps = pd.DataFrame(overlaps).sort_values('n_itens', ascending=False) if overlaps else pd.DataFrame()

print(f"Pares de grupos com sobreposição: {len(df_overlaps)}")
if len(df_overlaps):
    display(df_overlaps)

# ── Detalhe: alimentos concretos em cada par sobreposto ──────────────────
# ATENÇÃO ao revisar: sobreposição NÃO é necessariamente um bug. Grupos como
# 'ferro_heme'/'vitamina_a' ou 'ferro_nao_heme'/'vitamina_a' podem legitimamente
# compartilhar itens (ex.: fígado é fonte de ferro heme E vitamina A ao mesmo
# tempo -- isso é esperado). O que deve ser escrutinado são pares onde a
# sobreposição é SEMANTICAMENTE incoerente (ex.: 'doces' × 'legumes_verduras'
# ou 'doces' × 'vitamina_a' via 'batata-doce') -- sinal de colisão de substring,
# não de característica real do alimento.
for _, row in df_overlaps.iterrows():
    c1, c2 = f"flag_{row['grupo_1']}", f"flag_{row['grupo_2']}"
    itens = df_alimentos.loc[
        df_alimentos[c1] & df_alimentos[c2],
        [c for c in [col_alimento, 'categoria'] if c in df_alimentos.columns]
    ].drop_duplicates()
    print(f"\n── {row['grupo_1']} × {row['grupo_2']} ({row['n_itens']} itens) ──")
    display(itens)


Pares de grupos com sobreposição: 12


,grupo_1,grupo_2,n_itens
0,frutas_in_natura,vitamina_a,7
2,legumes_verduras,vitamina_a,6
1,legumes_verduras,ferro_nao_heme,4
7,ferro_nao_heme,vitamina_a,4
6,ferro_heme,conservas,2
4,ferro_heme,vitamina_a,2
3,legumes_verduras,doces,1
5,ferro_heme,produtos_carneos,1
8,ferro_nao_heme,conservas,1
9,vitamina_a,doces,1



── frutas_in_natura × vitamina_a (7 itens) ──


,alimento,categoria
83,"Cajá-Manga, cru",frutas
276,"Mamão, Formosa, cru",frutas
277,"Mamão, Papaia, cru",frutas
280,"Manga, Haden, crua",frutas
281,"Manga, Palmer, crua",frutas
283,"Manga, Tommy Atkins, crua",frutas
284,Mangaba,frutas



── legumes_verduras × vitamina_a (6 itens) ──


,alimento,categoria
64,"Batata, doce, crua",cereais_tuberculos
74,"Brócolis, cru",legumes_verduras
136,"Cenoura, crua",legumes_verduras
158,"Couve, manteiga, crua",legumes_verduras
159,"Couve-flor, crua",legumes_verduras
174,"Espinafre, Nova Zelândia, cru",legumes_verduras


... (16 outputs left out)


── produtos_carneos × conservas (1 itens) ──


,alimento,categoria
413,Salsicha em conserva,conservas



── biscoito_pao_bolo × doces (1 itens) ──


,alimento,categoria
343,Pão doce,biscoito_pao_bolo


## Bloco 5 — Construção dos parâmetros nutricionais por grupo/escola

Consolida, para cada combinação **escola × grupo nutricional × refeições/dia**, os limites
mínimos e máximos de cada nutriente que o solver deverá respeitar.


In [7]:
# ── Verificar colunas esperadas na demanda nutricional ────────────────────
colunas_esperadas_demanda = [
    'CO_ENTIDADE', 'grupo_nutricional', 'num_refeicoes', 'quantidade_alunos',
    'energia_kcal', 'proteina_min', 'proteina_max', 'lipidio_min', 'lipidio_max',
    'carbo_min', 'carbo_max', 'calcio', 'ferro', 'vitamina_a', 'vitamina_c',
]
colunas_faltantes = [c for c in colunas_esperadas_demanda if c not in df_demanda.columns]
if colunas_faltantes:
    print(f"⚠ Colunas não encontradas em demanda_nutricional_escolas.csv: {colunas_faltantes}")
    print("  Verifique se o módulo do censo foi executado com a versão mais recente.")
else:
    print("✓ Todas as colunas esperadas presentes na demanda nutricional.")

# ── Mapear grupo_nutricional → percentual nutricional da resolução ────────
# Mantido como METADADO (usado depois para escolher a diversidade mínima em
# montar_perfil()) — NÃO é mais usado para reponderar valores de nutrientes
# (ver correção abaixo).
MAPA_PERCENTUAL = {
    'creche_parcial'         : REGRAS_PNAE["percentual_nutricional_minimo"]["creche_parcial"],
    'creche_integral'        : REGRAS_PNAE["percentual_nutricional_minimo"]["creche_integral"],
    'indigena_quilombola_pct': REGRAS_PNAE["percentual_nutricional_minimo"]["indigena_quilombola_pct"],
    'parcial_1_refeicao'     : REGRAS_PNAE["percentual_nutricional_minimo"]["parcial_1_refeicao"],
    'parcial_2_refeicoes'    : REGRAS_PNAE["percentual_nutricional_minimo"]["parcial_2_refeicoes"],
    'integral'               : REGRAS_PNAE["percentual_nutricional_minimo"]["integral"],
}

def mapear_pct_nutricional(grupo_nutricional, num_refeicoes):
    """Mapeia grupo_nutricional x num_refeicoes para o percentual mínimo
    de cobertura das necessidades nutricionais diárias (Resolução nº 4/2026).
    Usado apenas como METADADO classificatório (ex.: diversidade mínima em
    montar_perfil()) — os valores nutricionais em si já vêm prontos do CSV.

    LIMITAÇÃO METODOLÓGICA DOCUMENTADA (set/2026): os microdados do Censo
    Escolar utilizados no Módulo 2 não trazem detalhamento suficiente para
    identificar quantos alunos são indígenas ou quilombolas por escola.
    Por isso, o ramo 'INDIGENA'/'QUILOMBOLA' abaixo é mantido no código por
    completude em relação à Resolução CD/FNDE nº 4/2026, mas NUNCA é
    acionado na prática — nenhum valor de 'grupo_nutricional' produzido
    pelo Módulo 2 atual contém essas strings (os grupos reais são CRECHE,
    PRE_ESCOLA, FUND_6_10, FUND_11_15, EJA, AEE). Convenção adotada neste
    projeto: "educação integral" é tratado como equivalente a "3 refeições
    ou mais" (num_refeicoes >= 3), que é o critério que efetivamente decide
    o percentual nutricional na função abaixo.
    """
    g = str(grupo_nutricional).upper()
    pn = REGRAS_PNAE["percentual_nutricional_minimo"]
    if 'CRECHE' in g:
        return pn["creche_integral"] if num_refeicoes >= 3 else pn["creche_parcial"]
    if 'INDIGENA' in g or 'QUILOMBOLA' in g:
        # Ramo não acionado com os dados atuais do Censo — ver limitação
        # documentada no docstring acima.
        return pn["indigena_quilombola_pct"]
    if num_refeicoes >= 3:
        return pn["integral"]  # "integral" == 3+ refeições/dia (convenção do projeto)
    if num_refeicoes == 2:
        return pn["parcial_2_refeicoes"]
    return pn["parcial_1_refeicao"]  # inclui 1 refeição/dia e AEE em contraturno

# ── Calcular limites mínimos e máximos de sódio por grupo ────────────────
def get_limite_sodio(grupo_nutricional, num_refeicoes):
    """Retorna limite de sódio (mg) conforme grupo e número de refeições.

    NOTA (set/2026): o teste `'integral' in grupo_nutricional` é vestigial —
    nenhum grupo_nutricional real (CRECHE, PRE_ESCOLA, FUND_6_10,
    FUND_11_15, EJA, AEE) contém a palavra "integral"; quem decide na
    prática é sempre `num_refeicoes >= 3` (mesma convenção adotada em
    mapear_pct_nutricional(), acima: "integral" == 3+ refeições/dia).
    Mantido por segurança/compatibilidade caso o Módulo 2 passe a nomear
    grupos de forma diferente no futuro.
    """
    ls = REGRAS_PNAE["limites_sodio"]
    if 'integral' in grupo_nutricional or num_refeicoes >= 3:
        return ls["integral_3_refeicoes"]["sodio_mg"]
    elif num_refeicoes >= 2:
        return ls["parcial_2_refeicoes"]["sodio_mg"]
    else:
        return ls["parcial_1_refeicao"]["sodio_mg"]

# ═══════════════════════════════════════════════════════════════════════════
# CORREÇÃO (jul/2026): demanda_nutricional_escolas.csv passou a ser gerado
# pelo Módulo Censo já com os VALORES-ALVO FINAIS por grupo_nutricional ×
# num_refeicoes (a % de cobertura da Resolução nº 4/2026 já foi aplicada
# upstream — confirmado com as tabelas de referência FNDE fornecidas pela
# autora: p.ex. CRECHE/3 refeições → energia_kcal=708, que bate exatamente
# com "70% das necessidades nutricionais/dia" da tabela FNDE, sem necessidade
# de nova multiplicação por pct).
#
# Causa-raiz do bug anterior: (1) este bloco multiplicava energia_kcal por
# `pct` novamente (ex.: 0.70 × 708 = 495,6 em vez de 708); (2) os nomes de
# coluna esperados (`proteina_g`, `lipideos_g`, `carboidratos_g`, `calcio_mg`,
# `ferro_mg`, `vitamina_a_mcg`, `vitamina_c_mg`) não batiam com os nomes reais
# do CSV (`proteina_min`/`proteina_max`, `lipidio_min`/`lipidio_max`,
# `carbo_min`/`carbo_max`, `calcio`, `ferro`, `vitamina_a`, `vitamina_c`), então
# nenhuma dessas colunas era gerada em df_params — e o solver.py caía em
# fórmulas de fallback genéricas (`pct * 15`, `pct * 300` etc.), descartando
# por completo os valores reais de referência FNDE por grupo/refeições.
#
# Correção: mapear diretamente as colunas reais do CSV para os nomes que o
# solver.py espera (min_<nutriente> / max_<nutriente>), sem qualquer
# multiplicação por pct. Proteína, lipídios e carboidratos já vêm com
# min/max prontos (intervalo de %VET já calculado pelo Censo); energia,
# cálcio, ferro, vitamina A e vitamina C vêm como valor mínimo único.
# ═══════════════════════════════════════════════════════════════════════════
MAPA_COLUNAS_NUTRIENTES = {
    # nome_usado_pelo_solver : (coluna_min_no_csv, coluna_max_no_csv_ou_None)
    'energia_kcal'   : ('energia_kcal', None),
    'proteina_g'     : ('proteina_min', 'proteina_max'),
    'carboidratos_g' : ('carbo_min', 'carbo_max'),
    'lipideos_g'     : ('lipidio_min', 'lipidio_max'),
    'calcio_mg'      : ('calcio', None),
    'ferro_mg'       : ('ferro', None),
    'vitamina_a_mcg' : ('vitamina_a', None),
    'vitamina_c_mg'  : ('vitamina_c', None),
}

linhas_params = []

for _, row in df_demanda.iterrows():
    grupo     = row.get('grupo_nutricional', '')
    n_ref     = int(row.get('num_refeicoes', 1))
    pct       = mapear_pct_nutricional(grupo, n_ref)   # metadado (diversidade)
    sodio_max = get_limite_sodio(grupo, n_ref)

    param = {
        'CO_ENTIDADE'       : row.get('CO_ENTIDADE'),
        'grupo_nutricional' : grupo,
        'num_refeicoes'     : n_ref,
        'quantidade_alunos' : row.get('quantidade_alunos', 0),
        'pct_nutricional'   : pct,
        'sodio_max_mg'      : sodio_max,
    }

    for nutriente, (col_min, col_max) in MAPA_COLUNAS_NUTRIENTES.items():
        if col_min in df_demanda.columns:
            v_min = row.get(col_min, np.nan)
            param[f'min_{nutriente}'] = float(v_min) if pd.notna(v_min) else None
        if col_max and col_max in df_demanda.columns:
            v_max = row.get(col_max, np.nan)
            param[f'max_{nutriente}'] = float(v_max) if pd.notna(v_max) else None

    linhas_params.append(param)

df_params = pd.DataFrame(linhas_params)

print(f"✓ Parâmetros nutricionais calculados: {len(df_params)} combinações escola × grupo")
print(f"  Colunas geradas: {[c for c in df_params.columns if c.startswith(('min_','max_'))]}")
display(df_params.head(10))

✓ Todas as colunas esperadas presentes na demanda nutricional.
✓ Parâmetros nutricionais calculados: 1021 combinações escola × grupo
  Colunas geradas: ['min_energia_kcal', 'min_proteina_g', 'max_proteina_g', 'min_carboidratos_g', 'max_carboidratos_g', 'min_lipideos_g', 'max_lipideos_g', 'min_calcio_mg', 'min_ferro_mg', 'min_vitamina_a_mcg', 'min_vitamina_c_mg']


,CO_ENTIDADE,grupo_nutricional,num_refeicoes,quantidade_alunos,pct_nutricional,sodio_max_mg,min_energia_kcal,min_proteina_g,max_proteina_g,min_carboidratos_g,max_carboidratos_g,min_lipideos_g,max_lipideos_g,min_calcio_mg,min_ferro_mg,min_vitamina_a_mcg,min_vitamina_c_mg
0,29179319,EJA,1,130,0.2,600,477.0,12.0,18.0,66.0,77.0,8.0,16.0,NaN,NaN,NaN,NaN
1,29179319,FUND_11_15,1,568,0.2,600,473.0,12.0,18.0,65.0,77.0,13.0,18.0,NaN,NaN,NaN,NaN
2,29179319,FUND_11_15,3,1,0.7,1400,1656.0,41.0,62.0,228.0,269.0,46.0,64.0,NaN,NaN,NaN,NaN
3,29179475,CRECHE,3,73,0.7,1400,708.0,18.0,27.0,97.0,115.0,20.0,28.0,350.0,2.0,147.0,9.0
4,29179475,PRE_ESCOLA,1,59,0.2,600,270.0,7.0,10.0,37.0,44.0,8.0,11.0,NaN,NaN,NaN,NaN
5,29179475,PRE_ESCOLA,3,1,0.7,1400,945.0,24.0,35.0,130.0,154.0,26.0,37.0,NaN,NaN,NaN,NaN
6,29179670,FUND_6_10,1,131,0.2,600,329.0,8.0,12.0,45.0,53.0,9.0,13.0,NaN,NaN,NaN,NaN
7,29179670,PRE_ESCOLA,3,15,0.7,1400,945.0,24.0,35.0,130.0,154.0,26.0,37.0,NaN,NaN,NaN,NaN
8,29180058,CRECHE,3,29,0.7,1400,708.0,18.0,27.0,97.0,115.0,20.0,28.0,350.0,2.0,147.0,9.0
9,29180058,PRE_ESCOLA,3,79,0.7,1400,945.0,24.0,35.0,130.0,154.0,26.0,37.0,NaN,NaN,NaN,NaN


## Bloco 6 — Restrições de frequência semanal

Gera a tabela de **limites mínimos e máximos de dias por semana** para cada grupo de alimento,
separada por tipo de período (parcial × integral).  
Esta tabela é usada diretamente pelo otimizador como variáveis de frequência.


In [8]:
# ── Tabela de restrições de frequência semanal ───────────────────────────
# Semana letiva = 5 dias

DIAS_SEMANA_LETIVA = 5

restricoes_frequencia = []

# 1. Frutas in natura
for periodo, dias_min in [
    ('parcial',  REGRAS_PNAE["flv_frequencia_minima_dias"]["frutas_in_natura_parcial"]),
    ('integral', REGRAS_PNAE["flv_frequencia_minima_dias"]["frutas_in_natura_integral"]),
]:
    restricoes_frequencia.append({
        'grupo'            : 'frutas_in_natura',
        'periodo'          : periodo,
        'tipo_restricao'   : 'MINIMO',
        'dias_semana'      : dias_min,
        'vezes_mes'        : None,
        'observacao'       : 'Art. 19 — oferta mínima semanal obrigatória',
    })

# 2. Legumes e verduras
for periodo, dias_min in [
    ('parcial',  REGRAS_PNAE["flv_frequencia_minima_dias"]["legumes_verduras_parcial"]),
    ('integral', REGRAS_PNAE["flv_frequencia_minima_dias"]["legumes_verduras_integral"]),
]:
    restricoes_frequencia.append({
        'grupo'            : 'legumes_verduras',
        'periodo'          : periodo,
        'tipo_restricao'   : 'MINIMO',
        'dias_semana'      : dias_min,
        'vezes_mes'        : None,
        'observacao'       : 'Art. 19 — oferta mínima semanal obrigatória',
    })

# 3. Ferro heme
restricoes_frequencia.append({
    'grupo'         : 'ferro_heme',
    'periodo'       : 'todos',
    'tipo_restricao': 'MINIMO',
    'dias_semana'   : REGRAS_PNAE["micronutrientes_frequencia_minima_dias"]["ferro_heme"],
    'vezes_mes'     : None,
    'observacao'    : 'Art. 19 — 4 dias/semana; se substituído por ferro não-heme, exige vit. C simultânea',
})

# 4. Vitamina A
restricoes_frequencia.append({
    'grupo'         : 'vitamina_a',
    'periodo'       : 'todos',
    'tipo_restricao': 'MINIMO',
    'dias_semana'   : REGRAS_PNAE["micronutrientes_frequencia_minima_dias"]["vitamina_a"],
    'vezes_mes'     : None,
    'observacao'    : 'Art. 19 — 3 dias/semana',
})

# 5. Limites máximos mensais convertidos para semana (÷ 4 semanas)
limites = REGRAS_PNAE["limites_maximos"]

maximos_mensais = [
    ('produtos_carneos',         'todos',    limites["produtos_carneos_vezes_mes"],          'MAXIMO_MES',  'Art. 18'),
    ('conservas',                'todos',    limites["conservas_vezes_mes"],                 'MAXIMO_MES',  'Art. 18'),
    ('lacteos_aditivados',       'parcial',  limites["lacteos_aditivados_parcial_vezes_mes"],'MAXIMO_MES',  'Art. 18'),
    ('lacteos_aditivados',       'integral', limites["lacteos_aditivados_integral_vezes_mes"],'MAXIMO_MES', 'Art. 18'),
    ('doces',                    'todos',    limites["doces_vezes_mes"],                     'MAXIMO_MES',  'Art. 18'),
    ('preparacoes_doces',        'parcial',  limites["preparacoes_doces_parcial_vezes_mes"], 'MAXIMO_MES',  'Art. 18'),
    ('margarina_creme_vegetal',  'parcial',  limites["margarina_parcial_vezes_mes"],         'MAXIMO_MES',  'Art. 18'),
]

maximos_semanais = [
    ('biscoito_pao_bolo', 'parcial_1ref',  limites["biscoito_pao_bolo_parcial_1ref_vezes_semana"],  'MAXIMO_SEM', 'Art. 18'),
    ('biscoito_pao_bolo', 'parcial_2ref',  limites["biscoito_pao_bolo_parcial_2ref_vezes_semana"],  'MAXIMO_SEM', 'Art. 18'),
    ('biscoito_pao_bolo', 'integral_3ref', limites["biscoito_pao_bolo_integral_3ref_vezes_semana"], 'MAXIMO_SEM', 'Art. 18'),
    ('preparacoes_doces', 'integral',      limites["preparacoes_doces_integral_vezes_semana"],      'MAXIMO_SEM', 'Art. 18'),
    ('margarina_creme_vegetal', 'integral',limites["margarina_integral_vezes_semana"],              'MAXIMO_SEM', 'Art. 18'),
]

for grupo, periodo, qtd, tipo, obs in maximos_mensais:
    restricoes_frequencia.append({
        'grupo'         : grupo,
        'periodo'       : periodo,
        'tipo_restricao': tipo,
        'dias_semana'   : None,
        'vezes_mes'     : qtd,
        'observacao'    : obs,
    })

for grupo, periodo, qtd, tipo, obs in maximos_semanais:
    restricoes_frequencia.append({
        'grupo'         : grupo,
        'periodo'       : periodo,
        'tipo_restricao': tipo,
        'dias_semana'   : qtd,
        'vezes_mes'     : None,
        'observacao'    : obs,
    })

df_freq = pd.DataFrame(restricoes_frequencia)

print(f"✓ Tabela de restrições de frequência: {len(df_freq)} regras")
display(df_freq)


✓ Tabela de restrições de frequência: 18 regras


,grupo,periodo,tipo_restricao,dias_semana,vezes_mes,observacao
0,frutas_in_natura,parcial,MINIMO,2.0,NaN,Art. 19 — oferta mínima semanal obrigatória
1,frutas_in_natura,integral,MINIMO,4.0,NaN,Art. 19 — oferta mínima semanal obrigatória
2,legumes_verduras,parcial,MINIMO,3.0,NaN,Art. 19 — oferta mínima semanal obrigatória
3,legumes_verduras,integral,MINIMO,5.0,NaN,Art. 19 — oferta mínima semanal obrigatória
4,ferro_heme,todos,MINIMO,4.0,NaN,Art. 19 — 4 dias/semana; se substituído por fe...
5,vitamina_a,todos,MINIMO,3.0,NaN,Art. 19 — 3 dias/semana
6,produtos_carneos,todos,MAXIMO_MES,NaN,2.0,Art. 18
7,conservas,todos,MAXIMO_MES,NaN,1.0,Art. 18
8,lacteos_aditivados,parcial,MAXIMO_MES,NaN,1.0,Art. 18
9,lacteos_aditivados,integral,MAXIMO_MES,NaN,2.0,Art. 18


## Bloco 7 — Restrição de Agricultura Familiar (45 %)

Calcula o **custo total de referência** dos alimentos e verifica a parcela oriunda
da Agricultura Familiar. Gera as variáveis de entrada para a restrição linear:

$$\sum_{j \in AF} c_j \cdot x_j \;\geq\; 0{,}45 \cdot \sum_{j} c_j \cdot x_j$$

onde $c_j$ é o preço de referência do alimento $j$ e $x_j$ é a quantidade adquirida.


In [9]:
# ── Parâmetros da restrição AF ───────────────────────────────────────────
af_params = REGRAS_PNAE["agricultura_familiar"]

PCT_MIN_AF   = af_params["percentual_meta_fnde_2026"]   # 0.45
PCT_LEGAL_AF = af_params["percentual_minimo_legal"]      # 0.45

print(f"Meta FNDE 2026 — Agricultura Familiar: {PCT_MIN_AF*100:.0f} %")
print(f"Mínimo legal (Lei nº 15.226/2025, altera Lei 11.947/2009): {PCT_LEGAL_AF*100:.0f} %")

# ── Verificar cobertura atual no dataset ──────────────────────────────────
# CORREÇÃO (jul/2026): a coluna de preço agora vem do join feito no Bloco 4
# ('preco_kg_referencia', originário do Módulo 3). O código antigo só procurava
# nomes de coluna já presentes em df_alimentos ANTES do join e por isso nunca
# encontrava preço nenhum (col_preco ficava sempre None).
col_preco = 'preco_kg_referencia' if 'preco_kg_referencia' in df_alimentos.columns else None

if col_preco is None:
    candidatos = [c for c in df_alimentos.columns if 'preco' in c.lower() or 'custo' in c.lower()]
    col_preco = candidatos[0] if candidatos else None
    if col_preco is None:
        print("⚠ Coluna de preço não localizada no dataset de alimentos.")
        print("  A restrição AF será gerada sem simulação de custo.")

# ═══════════════════════════════════════════════════════════════════════════
# AJUSTE (set/2026): diagnóstico de AF tornado "raw" (item 1.1 de auditoria)
# ─────────────────────────────────────────────────────────────────────────
# Causa-raiz do problema anterior: este bloco somava 'preco_kg_referencia'
# (R$/kg) de TODOS os alimentos do catálogo, SEM ponderação por quantidade
# demandada, e chamava isso de "percentual AF atual" com um veredito de
# status ("DENTRO DA META" / "ABAIXO DO MÍNIMO LEGAL"). Isso não é o
# percentual de gasto em AF — é a soma de preços unitários heterogêneos,
# que subestima estruturalmente o AF real (itens de AF tendem a ter preço/kg
# mais baixo). Confirmado: o AF real, calculado pelo solver com dados de
# compra reais (otimizacao_pnae_solver.py + gerar_relatorio_pnae.py, que
# calculam seu próprio pct_af a partir do resultado da otimização — nenhum
# dos dois lê os campos deste bloco), foi de 74,3 % para creche/março,
# contra 7,4 % deste cálculo bruto no mesmo dataset.
# Correção: manter os números brutos (informativos, para acompanhar a
# composição do catálogo elegível), mas remover o veredito de status e
# deixar explícito que isto NÃO é o percentual real de gasto em AF.
# Verificado (set/2026): nem otimizacao_pnae_solver.py nem
# gerar_relatorio_pnae.py leem 'restricao_agricultura_familiar',
# 'pct_af_diagnostico', 'status_diagnostico', 'coeficiente_min' ou
# 'coeficiente_legal' do restricoes_pnae.json — ambos os scripts calculam
# o AF real de forma independente, a partir da solução do modelo. Portanto,
# esta mudança não tem impacto downstream.
# ═══════════════════════════════════════════════════════════════════════════
if col_preco:
    df_alimentos['_preco_num'] = pd.to_numeric(df_alimentos[col_preco], errors='coerce').fillna(0)
    custo_total_catalogo = df_alimentos['_preco_num'].sum()
    custo_af_catalogo    = df_alimentos.loc[df_alimentos['agricultura_familiar'], '_preco_num'].sum()
    pct_af_atual = custo_af_catalogo / custo_total_catalogo if custo_total_catalogo > 0 else 0

    print(f"\n── Diagnóstico BRUTO de catálogo — Agricultura Familiar ────────────────")
    print("  ATENÇÃO: soma de preços de referência (R$/kg) de TODO o catálogo elegível,")
    print("  SEM ponderação por quantidade demandada. NÃO representa o percentual real")
    print("  de gasto em AF de um cardápio otimizado — esse valor só é conhecido após a")
    print("  resolução do modelo (Módulo 5). Uso: indicador bruto de composição do")
    print("  catálogo, não de conformidade.")
    print(f"  Soma dos preços de referência (catálogo)  : R$ {custo_total_catalogo:>12.2f}")
    print(f"  Soma dos preços de referência (AF)        : R$ {custo_af_catalogo:>12.2f}")
    print(f"  Cobertura bruta de catálogo (não é % real): {pct_af_atual*100:>8.1f} %")
else:
    pct_af_atual = None

# ── Objeto de restrição para o solver ────────────────────────────────────
restricao_af = {
    "tipo"                        : "agricultura_familiar",
    "descricao"                   : f"Mínimo de {PCT_MIN_AF*100:.0f} % do valor de aquisição deve ser de AF",
    "coeficiente_min"             : PCT_MIN_AF,
    "coeficiente_legal"           : PCT_LEGAL_AF,
    "flag_coluna"                 : "agricultura_familiar",   # coluna booleana no dataset de alimentos
    "base_calculo"                : "custo_total_aquisicao",
    "cobertura_bruta_catalogo_pct": pct_af_atual,
    "nota"                        : (
        "cobertura_bruta_catalogo_pct é a soma não ponderada de preços de referência "
        "do catálogo elegível (AF / total) — NÃO é o percentual real de gasto em AF de "
        "um cardápio otimizado. O AF real só é conhecido após a resolução do modelo "
        "(Módulo 5, otimizacao_pnae_solver.py), que calcula seu próprio percentual a "
        "partir das quantidades efetivamente adquiridas."
    ),
}

print(f"\n✓ Restrição AF estruturada para o solver.")
print(json.dumps(restricao_af, indent=2, ensure_ascii=False, default=str))


Meta FNDE 2026 — Agricultura Familiar: 45 %
Mínimo legal (Lei nº 15.226/2025, altera Lei 11.947/2009): 45 %

── Diagnóstico BRUTO de catálogo — Agricultura Familiar ────────────────
  ATENÇÃO: soma de preços de referência (R$/kg) de TODO o catálogo elegível,
  SEM ponderação por quantidade demandada. NÃO representa o percentual real
  de gasto em AF de um cardápio otimizado — esse valor só é conhecido após a
  resolução do modelo (Módulo 5). Uso: indicador bruto de composição do
  catálogo, não de conformidade.
  Soma dos preços de referência (catálogo)  : R$      9402.83
  Soma dos preços de referência (AF)        : R$       710.70
  Cobertura bruta de catálogo (não é % real):      7.6 %

✓ Restrição AF estruturada para o solver.
{
  "tipo": "agricultura_familiar",
  "descricao": "Mínimo de 45 % do valor de aquisição deve ser de AF",
  "coeficiente_min": 0.45,
  "coeficiente_legal": 0.45,
  "flag_coluna": "agricultura_familiar",
  "base_calculo": "custo_total_aquisicao",
  "cobertura_

## Bloco 8 — Dataset consolidado para o otimizador

Gera `alimentos_elegiveis_otimizador.csv` com TODAS as colunas necessárias para o modelo PL:
nutrientes, preço, flags de grupo, flag de AF e flag de restrição de oferta.


In [10]:
# ── Incorporar preço ao dataset de alimentos ──────────────────────────────
# CORREÇÃO (jul/2026): o join de preço/AF agora acontece uma única vez no
# Bloco 4 (célula de flags), então este bloco só precisa SELECIONAR as
# colunas — nenhum join redundante (e potencialmente divergente) é refeito aqui.

# ── Selecionar colunas relevantes para o otimizador ──────────────────────
# Colunas nutricionais presentes no dataset
cols_nutrientes = [c for c in df_alimentos.columns
                   if c in ['energia','proteina','lipideos','carboidratos','calcio',
                             'ferro','retinol','vitamina_c','sodio',
                             'energia_kcal','proteina_g','lipideos_g','carboidratos_g',
                             'calcio_mg','ferro_mg','vitamina_a_mcg','vitamina_c_mg','sodio_mg']]

# Colunas de flag de grupo operacional
cols_flags = [c for c in df_alimentos.columns if c.startswith('flag_')]

# Colunas obrigatórias de controle
cols_controle = [col_alimento, 'produto_busca','restricao_pnae', 'categoria', 'agricultura_familiar']
cols_controle = [c for c in cols_controle if c in df_alimentos.columns]

# Colunas de preço e rastreabilidade do match CONAB (Módulo 3)
cols_preco = [c for c in [col_preco, 'confianca_af', 'origem_regional_disponivel']
              if c and c in df_alimentos.columns]

# Colunas de origem e processamento (úteis para filtros no solver)
cols_extra = [c for c in ['origem', 'is_restrito', 'sdn_score', 'cluster_nutricional',
                            'dominancia_nutricional','is_ingrediente','motivo_ingrediente']
              if c in df_alimentos.columns]

colunas_finais = list(dict.fromkeys(
    cols_controle + cols_preco + cols_nutrientes + cols_flags + cols_extra
))

df_otimizador = df_alimentos[colunas_finais].copy()

# ═══════════════════════════════════════════════════════════════════════════
# CORREÇÃO: Deduplicação defensiva por 'produto_busca'
# ─────────────────────────────────────────────────────────────────────────
# Causa-raiz: O dataset trazia 8 registros duplicados por 'produto_busca' 
# (ex.: polpas de frutas e iogurte natural) devido a variações sutis de 
# espaços nas strings.
# Solução: Mantém o primeiro registro único por 'produto_busca' para evitar
# que o solver de Programação Linear receba variáveis duplicadas degeneradas.
# ═══════════════════════════════════════════════════════════════════════════
if 'produto_busca' in df_otimizador.columns:
    n_antes_dedup = len(df_otimizador)
    
    # 1. Normalização de espaços nos nomes dos alimentos
    df_otimizador['alimento'] = df_otimizador['alimento'].astype(str).str.strip()
    df_otimizador['produto_busca'] = df_otimizador['produto_busca'].astype(str).str.strip()
    
    # 2. Remoção de duplicatas mantendo a primeira ocorrência válida
    df_otimizador = df_otimizador.drop_duplicates(subset=['produto_busca'], keep='first').copy()
    
    n_removidos_dedup = n_antes_dedup - len(df_otimizador)
    if n_removidos_dedup > 0:
        print(f"✓ Deduplicação concluída: {n_removidos_dedup} alimento(s) duplicado(s) removido(s) por 'produto_busca'.")



# ── Remover alimentos de aquisição proibida (já feito no módulo anterior,
#    mas mantemos como salvaguarda) ──────────────────────────────────────
if 'restricao_pnae' in df_otimizador.columns:
    n_antes = len(df_otimizador)
    df_otimizador = df_otimizador[
        df_otimizador['restricao_pnae'] != 'Aquisição proibida'
    ].copy()
    n_removidos = n_antes - len(df_otimizador)
    if n_removidos:
        print(f"⚠ {n_removidos} alimentos de 'Aquisição proibida' removidos (salvaguarda).")

# ── Criar coluna de oferta_restrita (True = oferta limitada, inserir max freq) ──
if 'restricao_pnae' in df_otimizador.columns:
    df_otimizador['oferta_restrita'] = df_otimizador['restricao_pnae'].str.contains(
        'limitada', case=False, na=False
    )
else:
    df_otimizador['oferta_restrita'] = False

# ── Verificação de consistência (alimentos sem preço) ────────────────────
# AJUSTE (jul/2026): alimentos sem preço já são removidos no Bloco 4, logo
# n_sem_preco deve ser sempre 0 neste ponto. Mantido como rede de segurança:
# se aparecer > 0, indica que o Bloco 4 foi pulado/alterado, e o dado NÃO
# deve seguir para o Módulo 5 com preço ausente (nunca mais imputado por
# mediana de categoria — ver correção equivalente em otimizacao_pnae_solver.py).
if col_preco and col_preco in df_otimizador.columns:
    n_sem_preco = df_otimizador[col_preco].isna().sum()
    if n_sem_preco > 0:
        pct_sem = n_sem_preco / len(df_otimizador) * 100
        raise AssertionError(
            f"{n_sem_preco} alimentos ({pct_sem:.1f} %) seguem SEM preço de referência "
            f"neste ponto do pipeline — o filtro do Bloco 4 (linha "
            f"'preco_kg_referencia'.notna()) deveria ter removido todos eles. "
            f"Verifique se o Bloco 4 foi executado nesta sessão antes de prosseguir."
        )

# ── Reset de índice como ID único para o solver ──────────────────────────
df_otimizador = df_otimizador.reset_index(drop=True)
df_otimizador.index.name = 'id_alimento'
df_otimizador = df_otimizador.reset_index()

print(f"✓ Dataset do otimizador: {df_otimizador.shape[0]} alimentos × {df_otimizador.shape[1]} colunas")
print(f"  Alimentos com preço disponível: {df_otimizador[col_preco].notna().sum() if col_preco and col_preco in df_otimizador.columns else 'N/D'}")
print(f"  Alimentos de AF               : {df_otimizador['agricultura_familiar'].sum()}")
print(f"  Alimentos com oferta restrita : {df_otimizador['oferta_restrita'].sum()}")
display(df_otimizador.head())


✓ Deduplicação concluída: 8 alimento(s) duplicado(s) removido(s) por 'produto_busca'.
✓ Dataset do otimizador: 309 alimentos × 38 colunas
  Alimentos com preço disponível: 309
  Alimentos de AF               : 83
  Alimentos com oferta restrita : 55


,id_alimento,alimento,produto_busca,restricao_pnae,categoria,agricultura_familiar,preco_kg_referencia,energia,proteina,lipideos,carboidratos,calcio,ferro,retinol,vitamina_c,sodio,flag_frutas_in_natura,flag_legumes_verduras,flag_ferro_heme,flag_ferro_nao_heme,flag_vitamina_a,flag_produtos_carneos,flag_conservas,flag_lacteos_aditivados,flag_biscoito_pao_bolo,flag_doces,flag_preparacoes_doces,flag_margarina_creme_vegetal,flag_ultraprocessados,flag_acucar_simples_adicionado,origem,is_restrito,sdn_score,cluster_nutricional,dominancia_nutricional,is_ingrediente,motivo_ingrediente,oferta_restrita
0,0,"Abacate, cru",abacate,Sem restrição,frutas,False,4.2428,96.154709,1.239130,8.396667,6.030870,7.916667,0.206667,61.2,8.660000,0.000000,False,False,False,False,False,False,False,False,False,False,False,False,False,False,Vegetal,False,0.000174,2,-1.570170,False,NaN,False
1,1,"Abacaxi, cru",abacaxi,Sem restrição,frutas,False,3.8010,48.322213,0.858696,0.123333,12.334638,22.433333,0.256667,2.3,34.623333,0.000000,True,False,False,False,False,False,False,False,False,False,False,False,False,False,Vegetal,False,0.001088,2,-1.129063,False,NaN,False
2,2,"Abacaxi, polpa, congelada",abacaxi polpa,Sem restrição,sucos_bebidas,False,11.9000,30.591799,0.466667,0.113333,7.798667,13.538000,0.357333,2.0,1.246667,1.236333,False,False,False,False,False,False,False,False,False,False,False,False,False,False,Outros/Processados,False,0.000430,2,-1.726630,False,NaN,False
3,3,"Abóbora, cabotian, crua",abobora cabotian,Sem restrição,legumes_verduras,False,5.9900,40.000000,1.746377,0.536667,6.190000,18.000000,0.373333,508.0,5.090000,0.000000,False,False,False,False,True,False,False,False,False,False,False,False,False,False,Vegetal,False,0.000615,2,-1.418568,False,NaN,False
4,4,"Abóbora, menina brasileira, crua",abobora menina brasileira,Sem restrição,legumes_verduras,False,7.9900,13.000000,0.608696,0.000000,2.170000,8.740000,0.150000,1335.0,1.500000,0.000000,False,False,False,False,True,False,False,False,False,False,False,False,False,False,Vegetal,False,0.001481,2,-1.530574,False,NaN,False


## Bloco 9 — Auditoria: Checklist de completude da parametrização das regras

> **Escopo deste checklist (AJUSTE set/2026):** as verificações abaixo confirmam que os
> parâmetros da Resolução CD/FNDE nº 4/2026 foram **corretamente configurados no código**
> (`REGRAS_PNAE`) — ex.: "o limite de sódio tem 3 cenários definidos". Este checklist
> **NÃO avalia** se um cardápio ou o dataset de alimentos atualmente carregado **atende**
> a essas regras (isso é verificado depois, na etapa de otimização/solução do Módulo 5).
> Ver Bloco 7 para o diagnóstico (bruto, pré-otimização) da cobertura de Agricultura
> Familiar no dataset atual — os dois números não devem ser confundidos.

Verifica automaticamente se as restrições foram geradas corretamente e exibe o
**resumo executivo de completude da parametrização** no mesmo formato utilizado por
nutricionistas do PNAE.


In [11]:
# ═══════════════════════════════════════════════════════════════════════════
# AUDITORIA AUTOMÁTICA — Checklist Resolução CD/FNDE nº 4/2026
# ═══════════════════════════════════════════════════════════════════════════

auditoria = []

def check(item, condicao, detalhe=""):
    status = "✅" if condicao else "❌"
    auditoria.append({"item": item, "status": status, "detalhe": detalhe})
    print(f"{status} {item}  {detalhe}")

print("\n══════════════════════════════════════════════════════════════════════")
print("  CHECKLIST DE COMPLETUDE DA PARAMETRIZAÇÃO — Resolução CD/FNDE nº 4/2026")
print("  (audita o CÓDIGO, não a conformidade do cardápio/dataset — ver nota do Bloco 9)")
print("══════════════════════════════════════════════════════════════════════\n")

# 1. Percentual nutricional
check(
    "Percentual nutricional mínimo definido para todos os grupos",
    len(REGRAS_PNAE["percentual_nutricional_minimo"]) == 6,
    f"({len(REGRAS_PNAE['percentual_nutricional_minimo'])} grupos definidos)"
)

# 2. FLV — quantidade semanal
check(
    "Qtd. semanal FLV definida para parcial e integral",
    all(k in REGRAS_PNAE["flv_semana_gramas"] for k in ["parcial","integral"]),
    f"Parcial: {REGRAS_PNAE['flv_semana_gramas']['parcial']} g | Integral: {REGRAS_PNAE['flv_semana_gramas']['integral']} g"
)

# 3. Frutas — frequência mínima
check(
    "Frequência mínima de frutas in natura (2 dias parcial / 4 dias integral)",
    df_freq[df_freq['grupo']=='frutas_in_natura']['dias_semana'].tolist() == [2, 4],
    str(df_freq[df_freq['grupo']=='frutas_in_natura']['dias_semana'].tolist())
)

# 4. Legumes e verduras — frequência mínima
check(
    "Frequência mínima de legumes/verduras (3 dias parcial / 5 dias integral)",
    df_freq[df_freq['grupo']=='legumes_verduras']['dias_semana'].tolist() == [3, 5],
    str(df_freq[df_freq['grupo']=='legumes_verduras']['dias_semana'].tolist())
)

# 5. Ferro heme — frequência mínima
check(
    "Ferro heme ≥ 4 dias/semana",
    REGRAS_PNAE["micronutrientes_frequencia_minima_dias"]["ferro_heme"] == 4,
    f"Configurado: {REGRAS_PNAE['micronutrientes_frequencia_minima_dias']['ferro_heme']} dias"
)

# 6. Vitamina A — frequência mínima
check(
    "Vitamina A ≥ 3 dias/semana",
    REGRAS_PNAE["micronutrientes_frequencia_minima_dias"]["vitamina_a"] == 3,
    f"Configurado: {REGRAS_PNAE['micronutrientes_frequencia_minima_dias']['vitamina_a']} dias"
)

# 7. Limites máximos de frequência
check(
    "Limites de produtos cárneos, conservas, doces e ultraprocessados definidos",
    all(k in REGRAS_PNAE["limites_maximos"] for k in [
        "produtos_carneos_vezes_mes","conservas_vezes_mes","doces_vezes_mes"
    ]),
    "Carneos ≤2x/mês | Conservas ≤1x/mês | Doces ≤1x/mês"
)

# 8. Gordura trans
check(
    "Gordura trans industrializada proibida",
    REGRAS_PNAE["proibicoes"]["gordura_trans_industrializada"] is True,
)

# 9. Menores de 3 anos
check(
    "Proibições para menores de 3 anos (ultraprocessados, açúcar, mel, adoçantes)",
    all(REGRAS_PNAE["proibicoes"]["menores_3_anos"].values()),
)

# 10. Limites de sódio
check(
    "Limites de sódio definidos para 1, 2 e 3+ refeições",
    len(REGRAS_PNAE["limites_sodio"]) == 3,
    "600 mg / 800 mg / 1400 mg"
)

# 11. Diversidade mínima de alimentos
check(
    "Diversidade mínima de alimentos in natura por semana (10 / 14 / 23)",
    list(REGRAS_PNAE["diversidade_minima_alimentos"].values()) == [10, 14, 23],
)

# 12. Agricultura familiar
check(
    f"Mínimo da AF ≥ {PCT_MIN_AF*100:.0f} % configurado",
    REGRAS_PNAE["agricultura_familiar"]["percentual_meta_fnde_2026"] == PCT_MIN_AF,
)
# A REGRA é que o mínimo legal de AF deve ser aplicado no solver, 
# mas o percentual atual de AF no dataset pode ser menor o definido pelo FNDE 
# — o que não é um erro, apenas um diagnóstico. Por isso, a verificação do percentual 
# atual de AF é comentada abaixo.
# if pct_af_atual is not None:
#     check(
#         f"Dataset atual atende à meta AF de {PCT_MIN_AF*100:.0f} %",
#         pct_af_atual >= PCT_MIN_AF,
#         f"Atual: {pct_af_atual*100:.1f} %"
#     )

# 13. Bebida de fruta NÃO substitui fruta in natura
check(
    "Regra: bebida à base de fruta não substitui fruta in natura",
    REGRAS_PNAE["bebida_fruta_substitui_fruta_in_natura"] is False,
)

# 14. Ferro não-heme exige vitamina C simultânea
check(
    "Regra: substituição de ferro heme por não-heme exige vitamina C simultânea",
    REGRAS_PNAE["micronutrientes_frequencia_minima_dias"]["ferro_nao_heme_requer_vitamina_c"] is True,
)

print("\n══════════════════════════════════════════════════════════════════════")
n_ok  = sum(1 for a in auditoria if a['status'] == '✅')
n_nok = sum(1 for a in auditoria if a['status'] == '❌')
print(f"  RESULTADO: {n_ok} ✅ parâmetros corretamente configurados | {n_nok} ❌ pendentes")
print("  (isto NÃO afirma que o cardápio/dataset atual está em conformidade regulatória)")
print("══════════════════════════════════════════════════════════════════════")

df_auditoria = pd.DataFrame(auditoria)



══════════════════════════════════════════════════════════════════════
  CHECKLIST DE COMPLETUDE DA PARAMETRIZAÇÃO — Resolução CD/FNDE nº 4/2026
  (audita o CÓDIGO, não a conformidade do cardápio/dataset — ver nota do Bloco 9)
══════════════════════════════════════════════════════════════════════

✅ Percentual nutricional mínimo definido para todos os grupos  (6 grupos definidos)
✅ Qtd. semanal FLV definida para parcial e integral  Parcial: 280 g | Integral: 520 g
✅ Frequência mínima de frutas in natura (2 dias parcial / 4 dias integral)  [2.0, 4.0]
✅ Frequência mínima de legumes/verduras (3 dias parcial / 5 dias integral)  [3.0, 5.0]
✅ Ferro heme ≥ 4 dias/semana  Configurado: 4 dias
✅ Vitamina A ≥ 3 dias/semana  Configurado: 3 dias
✅ Limites de produtos cárneos, conservas, doces e ultraprocessados definidos  Carneos ≤2x/mês | Conservas ≤1x/mês | Doces ≤1x/mês
✅ Gordura trans industrializada proibida  
✅ Proibições para menores de 3 anos (ultraprocessados, açúcar, mel, adoçantes)  
✅ 

## Bloco 10 — Exportação dos outputs

Salva todos os artefatos de saída que o próximo módulo (Programação Linear) irá consumir.

| Arquivo | Formato | Usado pelo solver como |
|---|---|---|
| `restricoes_pnae.json` | JSON | Objeto completo de restrições |
| `restricoes_frequencia_pnae.csv` | CSV | Restrições de frequência em tabela |
| `parametros_nutricionais_solver.csv` | CSV | Limites mínimos/máximos por escola × grupo |
| `alimentos_elegiveis_otimizador.csv` | CSV | Dataset completo de alimentos para a PL |
| `relatorio_conformidade_pnae.csv` | CSV | Auditoria para nutricionistas/gestores |
| `relatorio_restricoes_pnae.txt` | TXT | Sumário legível para nutricionistas |


In [12]:
# ── 1. JSON completo de restrições ───────────────────────────────────────
restricoes_json = {
    **REGRAS_PNAE,
    "restricao_agricultura_familiar": restricao_af,
    "restricoes_frequencia"         : df_freq.to_dict(orient='records'),
}

with open(output_folder + 'restricoes_pnae.json', 'w', encoding='utf-8') as f:
    json.dump(restricoes_json, f, ensure_ascii=False, indent=2, default=str)
print("✓ restricoes_pnae.json")

# ── 2. Tabela de restrições de frequência ────────────────────────────────
df_freq.to_csv(
    output_folder + 'restricoes_frequencia_pnae.csv',
    index=False, encoding='utf-8-sig'
)
print("✓ restricoes_frequencia_pnae.csv")

# ── 3. Parâmetros nutricionais por escola × grupo ─────────────────────────
df_params.to_csv(
    output_folder + 'parametros_nutricionais_solver.csv',
    index=False, encoding='utf-8-sig'
)
print("✓ parametros_nutricionais_solver.csv")

# ── 4. Dataset consolidado para o otimizador ─────────────────────────────
df_otimizador.to_csv(
    output_folder + 'alimentos_elegiveis_otimizador.csv',
    index=False, encoding='utf-8-sig'
)
print("✓ alimentos_elegiveis_otimizador.csv")

# ── 5. Relatório de conformidade (auditoria) ─────────────────────────────
df_auditoria.to_csv(
    output_folder + 'relatorio_conformidade_pnae.csv',
    index=False, encoding='utf-8-sig'
)
print("✓ relatorio_conformidade_pnae.csv")

# ── 6. Relatório textual para nutricionistas ─────────────────────────────
linhas_relatorio = [
    "=" * 70,
    "RELATÓRIO DE RESTRIÇÕES PNAE — Módulo de Regras",
    f"Gerado em: {datetime.now().strftime('%d/%m/%Y %H:%M')}",
    f"Fonte legal: Resolução CD/FNDE nº 4, de 26 de fevereiro de 2026",
    "=" * 70,
    "",
    "1. PERCENTUAIS NUTRICIONAIS MÍNIMOS",
    "-" * 40,
]
for grupo, pct in REGRAS_PNAE["percentual_nutricional_minimo"].items():
    linhas_relatorio.append(f"  {grupo:35s}: {pct*100:.0f} %")

linhas_relatorio += [
    "",
    "2. FRUTAS, LEGUMES E VERDURAS (g/aluno/semana)",
    "-" * 40,
    f"  Período parcial : {REGRAS_PNAE['flv_semana_gramas']['parcial']} g",
    f"  Período integral: {REGRAS_PNAE['flv_semana_gramas']['integral']} g",
    "",
    "3. FREQUÊNCIA MÍNIMA SEMANAL DE MICRONUTRIENTES",
    "-" * 40,
    f"  Ferro heme  : {REGRAS_PNAE['micronutrientes_frequencia_minima_dias']['ferro_heme']} dias/semana",
    f"  Vitamina A  : {REGRAS_PNAE['micronutrientes_frequencia_minima_dias']['vitamina_a']} dias/semana",
    "",
    "4. LIMITES DE SÓDIO PER CAPITA",
    "-" * 40,
]
for cenario, vals in REGRAS_PNAE["limites_sodio"].items():
    linhas_relatorio.append(f"  {cenario:35s}: {vals['sodio_mg']} mg / {vals['sal_g']} g sal")

linhas_relatorio += [
    "",
    "5. AGRICULTURA FAMILIAR",
    "-" * 40,
    f"  Minimo legal (Lei 15.226/2025, altera Lei 11.947/2009): {REGRAS_PNAE['agricultura_familiar']['percentual_minimo_legal']*100:.0f} %",
    f"  Meta FNDE 2026 (adotada)       : {REGRAS_PNAE['agricultura_familiar']['percentual_meta_fnde_2026']*100:.0f} %",
]
if pct_af_atual is not None:
    linhas_relatorio.append(f"  Diagnostico atual do dataset (bruto, pre-otimizacao): {pct_af_atual*100:.1f} %")

linhas_relatorio += [
    "",
    "6. DIVERSIDADE MÍNIMA DE ALIMENTOS IN NATURA/MINIMAMENTE PROCESSADOS",
    "-" * 40,
    f"  1 refeição/dia (cobertura 20 %): {REGRAS_PNAE['diversidade_minima_alimentos']['cardapio_1_refeicao_ou_20pct']} alimentos diferentes/semana",
    f"  2 refeições/dia (cobertura 30%): {REGRAS_PNAE['diversidade_minima_alimentos']['cardapio_2_refeicoes_ou_30pct']} alimentos diferentes/semana",
    f"  3+ refeições/dia (cobertura 70%): {REGRAS_PNAE['diversidade_minima_alimentos']['cardapio_3_refeicoes_ou_70pct']} alimentos diferentes/semana",
    "",
    "=" * 70,
    "CHECKLIST DE AUDITORIA",
    "=" * 70,
]
for _, row in df_auditoria.iterrows():
    linhas_relatorio.append(f"  {row['status']} {row['item']}  {row['detalhe']}")

linhas_relatorio += [
    "",
    "=" * 70,
    "7. LIMITACOES METODOLOGICAS DOCUMENTADAS",
    "=" * 70,
    "  a) Indigenas/quilombolas: os microdados do Censo Escolar utilizados",
    "     no Modulo 2 nao trazem detalhamento suficiente para identificar",
    "     quantos alunos sao indigenas ou quilombolas por escola. Convencao",
    "     adotada: 'educacao integral' e tratado como equivalente a 3",
    "     refeicoes/dia ou mais.",
    "  b) Art. 19, II e III (gorduras totais e gordura saturada, % VET):",
    "     recomendacoes da Resolucao (nao proibicoes absolutas). NAO",
    "     implementadas como restricao ativa no solver -- a base de",
    "     composicao nutricional (TACO/TBCA) nao discrimina o tipo de",
    "     gordura por alimento.",
    "  c) Art. 19, I (acucar simples adicionado, recomendacao de <=7% VET):",
    "     nao calculado como %VET real. Identificado apenas por lista",
    "     fechada de ingredientes que SAO acucar/adocante adicionado puro",
    "     (acucar cristal, acucar refinado, acucar mascavo, mel de abelha,",
    "     melado, rapadura) -- ver flag 'acucar_simples_adicionado'.",
]
linhas_relatorio.append("")
linhas_relatorio.append(f"CHECKLIST DE COMPLETUDE DA PARAMETRIZACAO: {n_ok} conformes | {n_nok} nao conformes")
linhas_relatorio.append("(NAO representa conformidade do cardapio/dataset -- ver secao 9 do notebook)")
linhas_relatorio.append("=" * 70)

with open(output_folder + 'relatorio_restricoes_pnae.txt', 'w', encoding='utf-8') as f:
    f.write("\n".join(linhas_relatorio))
print("✓ relatorio_restricoes_pnae.txt")

print("\n── Resumo dos arquivos gerados ─────────────────────────────────────────")
for fname in [
    'restricoes_pnae.json',
    'restricoes_frequencia_pnae.csv',
    'parametros_nutricionais_solver.csv',
    'alimentos_elegiveis_otimizador.csv',
    'relatorio_conformidade_pnae.csv',
    'relatorio_restricoes_pnae.txt',
]:
    fpath = output_folder + fname
    if os.path.exists(fpath):
        size = os.path.getsize(fpath)
        print(f"  {fname:50s}: {size:>8} bytes")

print("\n✓ Módulo de Regras PNAE concluído. Todos os outputs prontos para o otimizador.")


✓ restricoes_pnae.json
✓ restricoes_frequencia_pnae.csv
✓ parametros_nutricionais_solver.csv
✓ alimentos_elegiveis_otimizador.csv
✓ relatorio_conformidade_pnae.csv
✓ relatorio_restricoes_pnae.txt

── Resumo dos arquivos gerados ─────────────────────────────────────────
  restricoes_pnae.json                              :    10488 bytes
  restricoes_frequencia_pnae.csv                    :     1207 bytes
  parametros_nutricionais_solver.csv                :    76811 bytes
  alimentos_elegiveis_otimizador.csv                :   107331 bytes
  relatorio_conformidade_pnae.csv                   :     1129 bytes
  relatorio_restricoes_pnae.txt                     :     4432 bytes

✓ Módulo de Regras PNAE concluído. Todos os outputs prontos para o otimizador.


## Bloco 11 — Guia de integração com o módulo de Programação Linear

O próximo módulo (PL) deverá consumir os outputs gerados aqui da seguinte forma:

```python
import json, pandas as pd

output_folder = './data/output/'

# 1. Carregar todas as restrições em um único objeto
with open(output_folder + 'restricoes_pnae.json') as f:
    restricoes = json.load(f)

# 2. Dataset de alimentos com flags prontas
df = pd.read_csv(output_folder + 'alimentos_elegiveis_otimizador.csv')

# 3. Parâmetros nutricionais por escola × grupo (linhas do solver)
df_params = pd.read_csv(output_folder + 'parametros_nutricionais_solver.csv')

# 4. Restrições de frequência semanal (para variáveis de decisão de cardápio)
df_freq = pd.read_csv(output_folder + 'restricoes_frequencia_pnae.csv')

# ── Uso típico no solver ──────────────────────────────────────────────────

# a) Meta de agricultura familiar
pct_af = restricoes['agricultura_familiar']['percentual_meta_fnde_2026']  # 0.45

# b) Flag de AF por alimento
mask_af = df['agricultura_familiar'] == True  # alimentos de AF

# c) Limite de sódio para um grupo específico
lim = restricoes['limites_sodio']['parcial_1_refeicao']['sodio_mg']  # 600 mg

# d) Mínimo de frutas (g/semana) para período parcial
min_frutas_g = restricoes['flv_semana_gramas']['parcial']  # 280 g

# e) Diversidade mínima para cardápio com 2 refeições
div_min = restricoes['diversidade_minima_alimentos']['cardapio_2_refeicoes_ou_30pct']  # 14
```

### Variáveis de decisão esperadas no modelo PL

| Variável | Tipo | Descrição |
|---|---|---|
| `x[j]` | Contínua ≥ 0 | Quantidade (g) do alimento `j` adquirida por aluno/dia |
| `y[j,d]` | Binária | Alimento `j` ofertado no dia `d` da semana (cardápio) |
| `z_af` | Derivada | Custo AF = $\sum_{j \in AF} c_j \cdot x_j$ |

### Restrições principais a implementar na PL

1. **Nutricionais (mínimo):** $\sum_j a_{nj} \cdot x_j \geq \text{min\_nutriente}_n$ para cada nutriente $n$
2. **Sódio (máximo):** $\sum_j \text{sodio}_j \cdot x_j \leq \text{sodio\_max}$
3. **FLV (mínimo semanal em g):** $\sum_{j \in FLV} \sum_d x_{jd} \geq 280$ (parcial)
4. **Frequência de grupos (mínimo dias):** $\sum_d y_{j,d} \geq \text{dias\_min}$ para $j \in$ grupo
5. **Limites de frequência (máximo):** $\sum_d y_{j,d} \leq \text{limite\_max}$ para grupos restritos
6. **Agricultura Familiar:** $\sum_{j \in AF} c_j x_j \geq 0.45 \cdot \sum_j c_j x_j$
7. **Diversidade:** número de alimentos distintos por semana $\geq$ diversidade\_min
8. **Proibições:** $x_j = 0$ para $j \in$ proibidos (flag `Aquisição proibida`)
